# MomentCut — long video to validated clips (free Colab)

This notebook runs the local part on Colab and leaves AI analysis to a provider
you choose.

The first cell clones the public repository on a fresh runtime and runs
`git pull --ff-only` when the checkout already exists, so updating does not
require uploading a project zip again.

- **Manual mode:** export chapter prompts, paste them into any AI, upload the
  JSON answers, then validate and render. No API key or API bill.
- **API mode:** optional; requires an OpenAI-compatible endpoint and may cost
  money. See the optional API section near the end.

The pipeline checks line IDs and quotes against the transcript before FFmpeg
ever sees a timestamp. Read `docs/COLAB.md` for assumptions, limits, and costs.

## 1. Clone or update the project

The next cell clones
[`ardyhim/momentcut`](https://github.com/ardyhim/momentcut) to
`/content/momentcut` on first run. On later runs in the same or a new runtime,
it runs `git pull --ff-only` so the code and notebook are current. No project
zip upload is needed.

- Change `REPO_URL` to use a fork.
- Set `UPDATE_REPO = False` to skip the automatic update for this session.
- Set `PROJECT_DIR` before this cell to use a Drive path.
- If the code was already imported and you pulled a change, restart the runtime
  before rerunning the later cells.
The clone updates the code under `/content/momentcut`. A Colab tab keeps its current cells, so reopen this notebook from GitHub when the notebook file itself changes.


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import zipfile

PROJECT_DIR = Path("/content/momentcut")
REPO_URL = "https://github.com/ardyhim/momentcut.git"
UPDATE_REPO = True

if not shutil.which("git"):
    raise SystemExit("git is required but was not found on PATH.")


def git(args):
    print("$ git", " ".join(args))
    return subprocess.run(["git", *args], check=True)


if (PROJECT_DIR / ".git").is_dir():
    if UPDATE_REPO:
        print("Updating existing checkout...")
        try:
            git(["-C", str(PROJECT_DIR), "pull", "--ff-only"])
        except subprocess.CalledProcessError:
            print(
                "[warn] git pull failed; continuing with the existing checkout. "
                "Resolve local changes or delete PROJECT_DIR to start fresh."
            )
elif PROJECT_DIR.exists():
    raise SystemExit(
        f"{PROJECT_DIR} exists but is not a git checkout. Move or delete it, or "
        "set PROJECT_DIR to another path."
    )
else:
    print("Cloning", REPO_URL)
    try:
        git(["clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)])
    except subprocess.CalledProcessError as exc:
        raise SystemExit(
            "git clone failed. Check network access and REPO_URL, then rerun this cell."
        ) from exc

if not (PROJECT_DIR / "momentcut").is_dir():
    raise SystemExit(f"MomentCut package not found in {PROJECT_DIR}.")

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))

head = subprocess.run(
    ["git", "-C", str(PROJECT_DIR), "rev-parse", "--short", "HEAD"],
    capture_output=True,
    text=True,
    check=False,
).stdout.strip()
print("Project:", PROJECT_DIR, "| commit:", head or "unknown")

## 2. Install dependencies

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
print("Installed requirements.")

import av
import faster_whisper

print("PyAV:", av.__version__, "| faster-whisper:", faster_whisper.__version__)
if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    raise SystemExit("FFmpeg/ffprobe is not on PATH.")
print("FFmpeg is available.")

## 3. Configure the run

`VIDEO_PATH` can point to an uploaded file or a Drive file. Leave it as `None`
to use the upload cell below.

Use `small` on a GPU, `base` on CPU, or `tiny`/an imported transcript when time
is tight. `TRANSCRIPT_PATH` skips ASR entirely if you already have
`.srt`, `.vtt`, or `.json`.

`RUN_DIR` lives under the checkout and is ignored by git, so generated videos
and analyses do not dirty the working tree.

In [ ]:
VIDEO_PATH = None          # or "/content/drive/MyDrive/video.mp4"
TRANSCRIPT_PATH = None     # optional: "/content/captions.srt"
RUN_DIR = PROJECT_DIR / "runs" / "colab-demo"
ASPECT = "vertical"        # "original" or "vertical"
SOURCE_OVERRIDE = None     # set current video path if the source moved

# Optional: persist model downloads on Drive.
# os.environ["MOMENTCUT_MODEL_DIR"] = "/content/drive/MyDrive/momentcut-models"


def has_nvidia_gpu():
    exe = shutil.which("nvidia-smi")
    if not exe:
        return False
    try:
        return subprocess.run(
            [exe, "-L"], capture_output=True, text=True, timeout=10
        ).returncode == 0
    except Exception:
        return False


GPU_AVAILABLE = has_nvidia_gpu()
ASR_MODEL = "small" if GPU_AVAILABLE else "base"
print("GPU detected:", GPU_AVAILABLE)
print("ASR model:", ASR_MODEL)
print("Run directory:", RUN_DIR)

### Upload the source video (skip if using a Drive path)

In [ ]:
if VIDEO_PATH is None:
    from google.colab import files

    print("Upload the source video...")
    uploaded = files.upload()
    video_suffixes = {".mp4", ".mov", ".mkv", ".webm", ".m4v"}
    candidates = [
        Path(name).resolve()
        for name in uploaded
        if Path(name).suffix.lower() in video_suffixes
    ]
    if not candidates:
        raise SystemExit("No supported video file was uploaded.")
    VIDEO_PATH = candidates[0]
else:
    VIDEO_PATH = Path(VIDEO_PATH)

if not VIDEO_PATH.exists():
    raise SystemExit(f"Video not found: {VIDEO_PATH}")
print("Video:", VIDEO_PATH)

## 4. Analyze and export prompts

This checks the source, transcribes with chunk checkpoints (or imports the
transcript), builds sentence lines and overlapping chapter blocks, and writes
portable prompts. It does not call an AI provider unless `--api` is added later.

In [ ]:
def run(args):
    import shlex

    print("$", " ".join(shlex.quote(str(item)) for item in args))
    result = subprocess.run([str(item) for item in args])
    if result.returncode:
        raise SystemExit(f"Command failed with exit code {result.returncode}")


analysis_args = [
    sys.executable,
    "-m",
    "momentcut",
    "analyze",
    str(VIDEO_PATH),
    "--run",
    str(RUN_DIR),
    "--asr-model",
    ASR_MODEL,
    "--asr-chunk-seconds",
    "600",
    "--min-duration",
    "15",
    "--max-duration",
    "90",
    "--max-clips",
    "8",
]
if TRANSCRIPT_PATH:
    analysis_args += ["--transcript", str(TRANSCRIPT_PATH)]

run(analysis_args)
print("Prompts:", RUN_DIR / "analysis" / "prompts")

## 5. Manual workflow: download prompts, paste into any AI

Download the prompt bundle, open each `block_XXX.md`, and paste its entire
contents into your provider. The prompt is self-contained. Save each answer as
a separate `.json`, `.txt`, or `.md` file.

Skip this loop if you are using the optional API section below.

In [ ]:
from google.colab import files
from IPython.display import Markdown, display

analysis_dir = RUN_DIR / "analysis"
prompt_zip = analysis_dir / "prompts.zip"
with zipfile.ZipFile(prompt_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (analysis_dir / "prompts").rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(analysis_dir))
    archive.write(analysis_dir / "blocks.json", "blocks.json")
files.download(str(prompt_zip))
print("Downloaded prompt bundle. Upload the answers in the next cell.")

### Upload provider answers

In [ ]:
from google.colab import files

print("Upload one JSON answer per block...")
uploaded = files.upload()
responses_dir = RUN_DIR / "analysis" / "responses"
responses_dir.mkdir(parents=True, exist_ok=True)
saved = 0
for name in uploaded:
    source = Path(name)
    if source.suffix.lower() not in {".json", ".txt", ".md"}:
        print("Skipping unsupported answer file:", name)
        continue
    destination = responses_dir / name
    shutil.copy(source, destination)
    saved += 1
    print("saved:", destination)
print("Saved", saved, "answer file(s).")

## 6. Validate, select, and inspect

Import reads saved answers, resolves line IDs and quotes, enforces duration and
source bounds, removes duplicates, and applies the score threshold. This stage
does not call an AI provider.

In [ ]:
from IPython.display import Markdown, display

import_args = [sys.executable, "-m", "momentcut", "import", str(RUN_DIR)]
if SOURCE_OVERRIDE:
    import_args += ["--source", str(SOURCE_OVERRIDE)]
run(import_args)
display(Markdown((RUN_DIR / "analysis" / "report.md").read_text(encoding="utf-8")))

## 7. Render and verify

In [ ]:
render_args = [
    sys.executable,
    "-m",
    "momentcut",
    "render",
    str(RUN_DIR),
    "--aspect",
    ASPECT,
]
if SOURCE_OVERRIDE:
    render_args += ["--source", str(SOURCE_OVERRIDE)]
run(render_args)

manifest = json.loads((RUN_DIR / "render_manifest.json").read_text(encoding="utf-8"))
for key, entry in manifest.get("clips", {}).items():
    if entry.get("status") == "rendered" and entry.get("aspect") == ASPECT:
        print(entry["id"], entry.get("duration"), "s ->", entry["path"])

## 8. Download the result

In [ ]:
clips_zip = RUN_DIR / "clips_bundle.zip"
with zipfile.ZipFile(clips_zip, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in (RUN_DIR / "clips").rglob("*"):
        if path.is_file():
            archive.write(path, path.relative_to(RUN_DIR))
    for name in ("analysis.json", "render_manifest.json"):
        path = RUN_DIR / name
        if path.exists():
            archive.write(path, name)
    report = RUN_DIR / "analysis" / "report.md"
    if report.exists():
        archive.write(report, "analysis_report.md")
files.download(str(clips_zip))
print("Downloaded:", clips_zip)

## 9. Optional API mode (may cost money)

Use this only if you have an OpenAI-compatible endpoint. The key is held only in
this runtime's environment. Completed blocks are reused on retry.

Examples of compatible services include OpenAI-compatible endpoints from
OpenAI, DeepSeek, OpenRouter, Groq, Together, or a local server. Check the
provider's current base URL, model name, pricing, rate limits, and content
policy; this project does not bundle access.

If you already imported manual answers, this will call only missing blocks.

In [ ]:
import getpass

from IPython.display import Markdown, display

os.environ["MOMENTCUT_API_BASE_URL"] = input("API base URL (e.g. https://.../v1): ").strip()
os.environ["MOMENTCUT_API_MODEL"] = input("Model name: ").strip()
os.environ["MOMENTCUT_API_KEY"] = getpass.getpass("API key (not echoed): ")

api_args = [
    sys.executable,
    "-m",
    "momentcut",
    "analyze",
    str(VIDEO_PATH),
    "--run",
    str(RUN_DIR),
    "--asr-model",
    ASR_MODEL,
    "--api",
]
if TRANSCRIPT_PATH:
    api_args += ["--transcript", str(TRANSCRIPT_PATH)]
run(api_args)
display(Markdown((RUN_DIR / "analysis" / "report.md").read_text(encoding="utf-8")))

## 10. If the runtime disconnects

- Prompts, answers, ASR chunks, analysis, and rendered clips are files in
  `RUN_DIR`. Re-running the cells reuses completed work.
- If the video path changed, set `SOURCE_OVERRIDE` to the new path and re-run
  import/render. The source fingerprint must match; different content is refused.
- If `/content` was deleted, the run directory is gone too. For long jobs, put
  `RUN_DIR` under Drive or download the artifacts after each stage.
- Re-running `analyze` after changing prompt-affecting settings archives old
  responses rather than mixing answers from different prompts. Re-importing the
  same answers is always free.

Troubleshooting is in `docs/COLAB.md`, and the exact answer contract is in
`docs/JSON_CONTRACT.md`.